# Board Game EDA — Exploratory Data Analysis (Updated Dataset)
## Optimizing Board Game Discovery Through Encoded Features and Representative Clustering

**By:** Santiago, Uy, Angos, Araña, Ramirez  
**Program:** BS Data Science — Asian Institute of Management

---

### Purpose of This Notebook

This notebook is the updated EDA using the new **`ttrpg_bgg_encoded_dataset.csv`** from the NEWFINAL folder.
Unlike the original dataset (name, description, average only), this dataset includes:
- `Number of Reviews` — a reliability signal for each score
- **159 binary category columns** (`Cat_*`) — what themes the game covers
- **192 binary mechanic columns** (`Mech_*`) — how the game is played

**What this notebook covers:**
1. Dataset loading, duplicate detection, and validation
2. Score distribution analysis
3. Class imbalance investigation (Hit / Average / Flop)
4. 10-point ordinal scale
5. Review count analysis
6. Text description analysis
7. Most frequent words
8. Top and bottom rated games
9. Category feature analysis (`Cat_*`)
10. Mechanics feature analysis (`Mech_*`)
11. Summary and key findings

**Charts are exported at 300 DPI** to the `revised eda charts/` folder.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import html as html_mod
import os

os.makedirs('revised eda charts', exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'

print('Libraries loaded!')

---
## 1. Load and Validate the Dataset

### Thought Process

The new dataset is a combined and encoded version of the BGG scrape. Before analyzing:
- **Duplicate detection** — multiple rows may represent the same game encoded under different categories
- **Missing values** — encoded binary columns could have NaN entries
- **Column inventory** — understand what `Cat_*` and `Mech_*` columns are present
- **Scale** — how many unique games after deduplication?

In [ ]:
# Load the dataset
df_raw = pd.read_csv('../../../data/new/NEWFINAL/ttrpg_bgg_encoded_dataset.csv')

cat_cols  = [c for c in df_raw.columns if c.startswith('Cat_')]
mech_cols = [c for c in df_raw.columns if c.startswith('Mech_')]

print(f'Raw dataset shape: {df_raw.shape}')
print(f'  Core columns:     4  (Name, Description, Average Score, Number of Reviews)')
print(f'  Category columns: {len(cat_cols)}')
print(f'  Mechanic columns: {len(mech_cols)}')
print(f'\n=== Core Column Types ===')
print(df_raw[['Name','Description','Average Score','Number of Reviews']].dtypes)
print(f'\n=== Missing Values (core columns) ===')
print(df_raw[['Name','Description','Average Score','Number of Reviews']].isnull().sum())
print(f'\n=== Duplicate Analysis ===')
full_dups = df_raw.duplicated().sum()
name_dups = df_raw.duplicated(subset='Name', keep=False).sum()
print(f'  Fully identical duplicate rows: {full_dups:,}')
print(f'  Rows sharing a game name:       {name_dups:,}')

# Drop exact duplicates for analysis
df = df_raw.drop_duplicates().reset_index(drop=True)
print(f'\nAfter dropping exact duplicates: {len(df):,} rows ({df["Name"].nunique():,} unique game names)')

### Finding 1: Dataset Has Significant Exact Duplicates

**Observations:**
- The raw CSV contains 10,000 rows but ~6,000 are exact duplicates of other rows.
- After dropping duplicates, ~4,883 unique game records remain.
- This is consistent with how the dataset was assembled: some games were scraped multiple times across category/mechanic joins.

**Decision:** All subsequent analysis uses the deduplicated dataset (`df`). The duplicate removal is lossless — no unique data is discarded.

**Note:** A small number of games (~52) still appear more than once after deduplication — these have the same name but different encoded values, suggesting variant editions or different BGG entries.

---
## 2. Understanding the Score Distribution

### Thought Process

`Average Score` is our target variable. Before modeling, we need to understand:
- What does the distribution look like? Normal? Skewed? Bimodal?
- What's the center and spread?
- How does this compare to the original BGG dataset (mean=6.60, std=0.81)?

In [ ]:
print('=== Average Score Statistics ===')
print(df['Average Score'].describe().round(4))
print(f'\nSkewness: {df["Average Score"].skew():.4f}')
print(f'Kurtosis: {df["Average Score"].kurtosis():.4f}')

In [ ]:
# CHART 1: Score Distribution Histogram
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(df['Average Score'], bins=50, color='#0D9488', edgecolor='white', alpha=0.9)
ax.axvline(df['Average Score'].mean(), color='#EF4444', linestyle='--', linewidth=2,
           label=f'Mean: {df["Average Score"].mean():.2f}')
ax.axvline(df['Average Score'].median(), color='#F59E0B', linestyle='--', linewidth=2,
           label=f'Median: {df["Average Score"].median():.2f}')

ax.set_xlabel('Average Score', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title('Distribution of Average Review Scores (Deduplicated)', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig('revised eda charts/N01_score_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

### Finding 2: Score Distribution Is Wider Than the Original Dataset

**Observations:**
- Mean ~6.59, std ~1.58 — compared to original BGG dataset (mean=6.60, std=0.81)
- The new dataset has a much wider spread, with scores spanning the full 1–10 range
- This is expected: the original dataset filtered to 10,000 top-reviewed games (biasing toward a narrow middle band); the new dataset includes games with very few reviews, which can have extreme scores

**Decision:** The wider distribution actually benefits the model — there's more signal differentiation between high and low quality games.

---
## 3. Defining and Analyzing the Target Labels

### Thought Process

Using the same labeling strategy as the original notebook for direct comparison:
- **Hit:** 8.0+ — exceptional on BGG
- **Average:** 6.0–7.9 — decent but not remarkable
- **Flop:** Below 6.0 — below hobby community threshold

In [ ]:
def label_game(score):
    if score >= 8.0:
        return 'Hit (8-10)'
    elif score >= 6.0:
        return 'Average (6-7.9)'
    else:
        return 'Flop (<6)'

df['label'] = df['Average Score'].apply(label_game)

# CHART 2: Class Distribution
fig, ax = plt.subplots(figsize=(8, 6))

labels_order = ['Hit (8-10)', 'Average (6-7.9)', 'Flop (<6)']
counts = [df[df['label'] == l].shape[0] for l in labels_order]
colors = ['#10B981', '#3B82F6', '#EF4444']

bars = ax.bar(labels_order, counts, color=colors, edgecolor='white', width=0.6)
for bar, count in zip(bars, counts):
    pct = count / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{count:,}\n({pct:.1f}%)', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Count', fontsize=13)
ax.set_title('Class Distribution: Hit vs Average vs Flop', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/N02_class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print('Class breakdown:')
for label in labels_order:
    count = (df['label'] == label).sum()
    print(f'  {label}: {count:,} ({count/len(df)*100:.1f}%)')

print('\nComparison with original dataset:')
print('  Original: Hit=340 (3.4%), Average=7608 (76.1%), Flop=2052 (20.5%)')

### Finding 3: Class Imbalance Remains but Is Less Extreme

**Observations:**
- The new dataset has a more balanced class distribution than the original
- The Flop class is now larger (more low-rated games in the dataset)
- The Hit class still represents a minority but has grown proportionally

**Why this changed:** The original BGG dataset was curated (top 10,000 most-reviewed games), which naturally excluded low-rated games. The new dataset includes games with few reviews, many of which score at extremes.

**Implication:** Class imbalance is still a concern, but the wider distribution makes ordinal regression a stronger choice than in the original setup.

---
## 4. The 10-Point Ordinal Scale

### Thought Process

The same reasoning as the original notebook applies: rounding scores to the nearest integer creates a more balanced distribution and allows ordinal regression instead of 3-class classification.

In [ ]:
df['score_10pt'] = df['Average Score'].round().astype(int).clip(1, 10)

# CHART 3: 10-Point Ordinal Distribution
fig, ax = plt.subplots(figsize=(10, 6))

bin_counts = df['score_10pt'].value_counts().sort_index()
ax.bar(bin_counts.index, bin_counts.values, color='#8B5CF6', edgecolor='white')
for idx, val in bin_counts.items():
    ax.text(idx, val + 15, str(val), ha='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Score (Rounded to Integer)', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title('10-Point Ordinal Scale Distribution', fontsize=15, fontweight='bold')
ax.set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig('revised eda charts/N03_10point_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

props = df['score_10pt'].value_counts(normalize=True)
baseline_acc = (props ** 2).sum()
print(f'10-point baseline accuracy (most-frequent class): {props.max():.4f}')
print(f'10-point baseline RMSE estimate: {df["Average Score"].std():.3f}')
print(f'\n10-point distribution:')
for score, count in bin_counts.items():
    print(f'  Score {score}: {count:,} games ({count/len(df)*100:.1f}%)')

### Finding 4: 10-Point Scale Is Well Distributed

**Observations:**
- The new dataset shows a more uniform spread across the 1–10 scale than the original
- Scores 5–8 have the highest representation
- Scores 1–2 and 9–10 are rare but present

**Decision:** The 10-point ordinal scale remains the correct framing. Predict continuous scores → round to this scale → evaluate with RMSE.

---
## 5. Review Count Analysis

### Thought Process

The new dataset includes `Number of Reviews`, which the original BGG dataset lacked. This lets us investigate:
- Do games with fewer reviews have noisier/more extreme scores?
- Is there a review threshold below which scores are unreliable?
- How does this affect our modeling strategy?

In [ ]:
print('=== Review Count Statistics ===')
print(df['Number of Reviews'].describe().round(2))

# CHART 4: Review Count Distribution + Score vs Reviews
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: distribution (log scale)
axes[0].hist(df['Number of Reviews'].clip(upper=500), bins=60,
             color='#0D9488', edgecolor='white', alpha=0.9)
axes[0].set_xlabel('Number of Reviews (capped at 500)', fontsize=13)
axes[0].set_ylabel('Count', fontsize=13)
axes[0].set_title('Distribution of Review Counts', fontsize=14, fontweight='bold')

# Right: score vs review count
sample = df.sample(min(2000, len(df)), random_state=42)
sc_colors = sample['label'].map({'Hit (8-10)': '#10B981', 'Average (6-7.9)': '#3B82F6', 'Flop (<6)': '#EF4444'})
axes[1].scatter(sample['Number of Reviews'], sample['Average Score'],
                alpha=0.3, s=15, c=sc_colors)
axes[1].set_xscale('log')
axes[1].set_xlabel('Number of Reviews (log scale)', fontsize=13)
axes[1].set_ylabel('Average Score', fontsize=13)
axes[1].set_title('Score vs. Number of Reviews', fontsize=14, fontweight='bold')

from matplotlib.lines import Line2D
legend_el = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#10B981', markersize=8, label='Hit'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#3B82F6', markersize=8, label='Average'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#EF4444', markersize=8, label='Flop'),
]
axes[1].legend(handles=legend_el, fontsize=11)

plt.tight_layout()
plt.savefig('revised eda charts/N04_reviews_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Score reliability by review bracket
brackets = [(1, 5), (6, 20), (21, 100), (101, 1000), (1001, 200000)]
print('\nScore statistics by review count bracket:')
print(f'{"Reviews":>14s}  {"Count":>6s}  {"Mean":>6s}  {"Std":>6s}  {"Min":>5s}  {"Max":>5s}')
for low, high in brackets:
    subset = df[(df['Number of Reviews'] >= low) & (df['Number of Reviews'] <= high)]
    if len(subset) > 0:
        print(f'  {low:>5d}-{high:<6d} {len(subset):>6,}  '
              f'{subset["Average Score"].mean():>6.2f}  {subset["Average Score"].std():>6.2f}  '
              f'{subset["Average Score"].min():>5.2f}  {subset["Average Score"].max():>5.2f}')

### Finding 5: Low-Review Games Have Noisier Scores

**Observations:**
- Games with 1–5 reviews have the widest score variance — a single reviewer can push the score to 1 or 10
- Games with 100+ reviews converge to a tighter, more reliable score band
- The scatter plot shows that very high scores (9–10) and very low scores (1–2) are mostly found among low-review games

**Decision:** For robust model training, consider weighting samples by review count or filtering to games with a minimum review threshold. Text-based features (TF-IDF on descriptions) will be more stable than review-derived scores for low-review games.

---
## 6. Text Description Analysis

### Thought Process

Descriptions remain our primary NLP input. We verify:
- Are descriptions long enough for TF-IDF?
- Do Hit games have longer descriptions?
- How does description length compare to the original dataset (mean=207 words)?

In [ ]:
def clean_text(text):
    text = html_mod.unescape(str(text))
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_desc'] = df['Description'].apply(clean_text)
df['word_count']  = df['clean_desc'].apply(lambda x: len(x.split()))

print('=== Description Length Statistics ===')
print(f'Mean word count:   {df["word_count"].mean():.0f} words')
print(f'Median word count: {df["word_count"].median():.0f} words')
print(f'Min word count:    {df["word_count"].min()} words')
print(f'Max word count:    {df["word_count"].max()} words')
print(f'\nComparison with original BGG dataset: mean=207, median=173')

In [ ]:
# CHART 5: Word Count Distribution + Word Count by Class
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(df['word_count'], bins=50, color='#F59E0B', edgecolor='white', alpha=0.9)
axes[0].axvline(df['word_count'].mean(), color='#EF4444', linestyle='--', linewidth=2,
                label=f'Mean: {df["word_count"].mean():.0f} words')
axes[0].set_xlabel('Word Count', fontsize=13)
axes[0].set_ylabel('Number of Games', fontsize=13)
axes[0].set_title('Distribution of Description Lengths', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)

labels_order = ['Hit (8-10)', 'Average (6-7.9)', 'Flop (<6)']
colors = ['#10B981', '#3B82F6', '#EF4444']
means_wc = [df[df['label'] == l]['word_count'].mean() for l in labels_order]
bars = axes[1].bar(labels_order, means_wc, color=colors, edgecolor='white', width=0.6)
for bar, m in zip(bars, means_wc):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                f'{m:.0f}', ha='center', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Average Word Count', fontsize=13)
axes[1].set_title('Average Description Length by Class', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/N05_wordcount_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nAverage word count by class:')
for l, m in zip(labels_order, means_wc):
    print(f'  {l}: {m:.0f} words')

### Finding 6: Descriptions Remain Sufficient for NLP

**Observations:**
- The average description length is comparable to the original dataset — sufficient for TF-IDF
- Hit games still tend to have longer descriptions, suggesting that complexity and quality are correlated
- The wide word count range (short to very long) means TF-IDF will encounter varying vector densities

**Decision:** TF-IDF on descriptions remains a viable feature source. The pattern of Hit games having longer descriptions is preserved in the new dataset.

---
## 7. Most Frequent Words in Descriptions

### Thought Process

Identifying dominant vocabulary to determine if custom stopwords are needed and whether the thematic content changed between the old and new datasets.

In [ ]:
stopwords = set('''
the a an and or but in on at to for of with by from is it that this are was were
be been being have has had do does did will would could should may might can shall not no their they
them he she his her its we our you your each all any some one two three four five more most other
than then when which who what where how if as up out about into over after before between under
again further there here also very just only own same so too such both few many much new old first
last long great little man back even still way take come make like time get go see know need want
use find give tell work call try ask put keep let set play run move live believe hold bring happen
write provide sit stand lose pay meet include continue show next without enough well through during
off down those these since while now per another every must upon game games player players card cards
turn turns board piece pieces point points round rounds end different using used based order number
place action actions hand rules rule side world team however able become part around made possible
winning win won among sets takes taken starting started along across always already often usually
sometimes never rather whether either neither yet least instead unless except within second third
everything nothing something anything everyone anyone someone else
'''.split())

all_words = ' '.join(df['clean_desc']).split()
filtered  = [w for w in all_words if w not in stopwords and len(w) > 2]
word_freq = Counter(filtered).most_common(15)

# CHART 6: Top 15 Words
fig, ax = plt.subplots(figsize=(10, 6))
words, cnts = zip(*word_freq)
ax.barh(range(len(words)-1, -1, -1), cnts, color='#0D9488', edgecolor='white')
ax.set_yticks(range(len(words)-1, -1, -1))
ax.set_yticklabels(words, fontsize=12)
ax.set_xlabel('Frequency', fontsize=13)
ax.set_title('Top 15 Most Frequent Words in Descriptions', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/N06_top_words.png', dpi=300, bbox_inches='tight')
plt.show()

print('Top 15 words:')
for word, count in word_freq:
    print(f'  {word}: {count:,}')

### Finding 7: Vocabulary Remains Rich in Thematic and Mechanic Terms

**Observations:**
- After removing stopwords and generic game terms, the remaining vocabulary reflects themes (fantasy, adventure, war, horror) and mechanics (combat, deck, tokens, dice)
- The new dataset vocabulary overlaps heavily with the original, confirming consistent data quality
- Words like 'character', 'adventure', 'scenario' may appear more prominently due to the TTRPG component of the combined dataset

**Decision:** The same custom stopword list from the original notebook remains appropriate. TF-IDF will capture game-specific discriminative vocabulary.

---
## 8. Highest and Lowest Rated Games

### Thought Process

Looking at extremes to validate that the new dataset's scores reflect genuine quality. We filter to games with 10+ reviews to avoid single-reviewer outliers.

In [ ]:
df_reliable = df[df['Number of Reviews'] >= 10].copy()
print(f'Games with 10+ reviews: {len(df_reliable):,} out of {len(df):,}')

# CHART 7: Top and Bottom Games
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top10 = df_reliable.nlargest(10, 'Average Score')[['Name', 'Average Score', 'Number of Reviews']]
bot10 = df_reliable.nsmallest(10, 'Average Score')[['Name', 'Average Score', 'Number of Reviews']]

axes[0].barh(range(9, -1, -1), top10['Average Score'].values, color='#10B981', edgecolor='white')
axes[0].set_yticks(range(9, -1, -1))
axes[0].set_yticklabels([n[:35] for n in top10['Name'].values], fontsize=9)
axes[0].set_xlabel('Average Score', fontsize=12)
axes[0].set_title('Top 10 Highest Rated Games (10+ reviews)', fontsize=13, fontweight='bold')
for i, (_, row) in enumerate(top10.iterrows()):
    axes[0].text(row['Average Score'] + 0.02, 9 - i,
                f'{row["Average Score"]:.2f} ({int(row["Number of Reviews"])}r)',
                va='center', fontsize=8)

axes[1].barh(range(9, -1, -1), bot10['Average Score'].values, color='#EF4444', edgecolor='white')
axes[1].set_yticks(range(9, -1, -1))
axes[1].set_yticklabels([n[:35] for n in bot10['Name'].values], fontsize=9)
axes[1].set_xlabel('Average Score', fontsize=12)
axes[1].set_title('Top 10 Lowest Rated Games (10+ reviews)', fontsize=13, fontweight='bold')
for i, (_, row) in enumerate(bot10.iterrows()):
    axes[1].text(row['Average Score'] + 0.05, 9 - i,
                f'{row["Average Score"]:.2f} ({int(row["Number of Reviews"])}r)',
                va='center', fontsize=8)

plt.tight_layout()
plt.savefig('revised eda charts/N07_top_bottom_games.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nTop 5 Highest Rated:')
for _, row in top10.head().iterrows():
    print(f'  {row["Name"]}: {row["Average Score"]:.2f} ({int(row["Number of Reviews"])} reviews)')

print('\nTop 5 Lowest Rated:')
for _, row in bot10.head().iterrows():
    print(f'  {row["Name"]}: {row["Average Score"]:.2f} ({int(row["Number of Reviews"])} reviews)')

### Finding 8: Score Extremes Are Consistent With BGG Community Preferences

**Observations:**
- **Highest-rated games** are complex, niche hobby titles with deep mechanics — consistent with the original dataset
- **Lowest-rated games** include mass-market or novelty titles that the BGG community considers too simple
- The 10+ review filter ensures these aren't noise from single reviewers

**Platform bias reminder:** This model learns hobby gamer preferences (BGG audience), not general public preferences. That's appropriate for specialty game retailers, not mass-market shelves.

---
## 9. Category Feature Analysis (`Cat_*` columns)

### Thought Process

The new dataset includes 159 binary category features. These are **structured features** that the original dataset lacked. Key questions:
- Which categories are most common?
- Do certain categories correlate with higher scores?
- How concentrated are the categories (many games in few categories, or evenly spread)?

In [ ]:
cat_sums = df[cat_cols].sum().sort_values(ascending=False)

print('Category coverage:')
print(f'  Total categories: {len(cat_cols)}')
print(f'  Non-zero categories: {(cat_sums > 0).sum()}')
print(f'  Avg categories per game: {df[cat_cols].sum(axis=1).mean():.2f}')
print(f'  Max categories on one game: {df[cat_cols].sum(axis=1).max()}')
print(f'\nTop 20 categories by game count:')
print(cat_sums.head(20).to_string())

In [ ]:
# CHART 8: Top 20 Categories by game count
fig, ax = plt.subplots(figsize=(10, 8))
top_cats = cat_sums.head(20)
clean_names = [c.replace('Cat_', '') for c in top_cats.index]
ax.barh(range(19, -1, -1), top_cats.values, color='#3B82F6', edgecolor='white')
ax.set_yticks(range(19, -1, -1))
ax.set_yticklabels(clean_names, fontsize=10)
ax.set_xlabel('Number of Games', fontsize=13)
ax.set_title('Top 20 Most Common Game Categories', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/N08_top_categories.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Average score per top category
top15_cats = cat_sums.head(15).index.tolist()
cat_score_data = []
for col in top15_cats:
    sub = df[df[col] == 1]
    cat_score_data.append({
        'Category': col.replace('Cat_', ''),
        'Count': len(sub),
        'Mean Score': sub['Average Score'].mean(),
        'Std Score': sub['Average Score'].std()
    })

cat_df = pd.DataFrame(cat_score_data).sort_values('Mean Score', ascending=False)

# CHART 9: Average score by top category
fig, ax = plt.subplots(figsize=(10, 7))
colors_cat = ['#10B981' if s >= 7 else '#3B82F6' if s >= 6 else '#EF4444'
              for s in cat_df['Mean Score']]
bars = ax.barh(range(len(cat_df)-1, -1, -1), cat_df['Mean Score'].values,
               color=colors_cat, edgecolor='white')
ax.set_yticks(range(len(cat_df)-1, -1, -1))
ax.set_yticklabels(cat_df['Category'].values, fontsize=10)
ax.axvline(df['Average Score'].mean(), color='gray', linestyle='--', linewidth=1.5,
           label=f'Overall mean: {df["Average Score"].mean():.2f}')
ax.set_xlabel('Average Score', fontsize=13)
ax.set_title('Average Score by Category (Top 15 Most Common)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('revised eda charts/N09_score_by_category.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nAverage score by category:')
print(cat_df[['Category', 'Count', 'Mean Score']].to_string(index=False))

### Finding 9: Category Features Carry Predictive Signal

**Observations:**
- **Fantasy and Adventure** games dominate the dataset — these are the primary themes in the hobby space
- Average scores vary meaningfully by category — some categories consistently score above or below the overall mean
- Complex, thematic categories (War games, Miniatures) tend to score higher than casual/mass-market categories

**Why this matters:** Category binary features are a new, structured signal that the original model lacked. Including these as features alongside TF-IDF could improve predictive accuracy, especially for games with short descriptions.

---
## 10. Mechanics Feature Analysis (`Mech_*` columns)

### Thought Process

The 192 mechanic columns capture *how* each game is played. Mechanics tend to be more specific than categories and may be stronger predictors of quality (a game with 'Cooperative Game' + 'Legacy Game' mechanics signals a specific, high-investment design).

In [ ]:
mech_sums = df[mech_cols].sum().sort_values(ascending=False)

print('Mechanic coverage:')
print(f'  Total mechanics: {len(mech_cols)}')
print(f'  Non-zero mechanics: {(mech_sums > 0).sum()}')
print(f'  Avg mechanics per game: {df[mech_cols].sum(axis=1).mean():.2f}')
print(f'  Max mechanics on one game: {df[mech_cols].sum(axis=1).max()}')
print(f'\nTop 20 mechanics by game count:')
print(mech_sums.head(20).to_string())

In [ ]:
# CHART 10: Top 20 Mechanics by game count
fig, ax = plt.subplots(figsize=(10, 8))
top_mechs = mech_sums.head(20)
clean_mech_names = [c.replace('Mech_', '')[:40] for c in top_mechs.index]
ax.barh(range(19, -1, -1), top_mechs.values, color='#8B5CF6', edgecolor='white')
ax.set_yticks(range(19, -1, -1))
ax.set_yticklabels(clean_mech_names, fontsize=10)
ax.set_xlabel('Number of Games', fontsize=13)
ax.set_title('Top 20 Most Common Game Mechanics', fontsize=15, fontweight='bold')

plt.tight_layout()
plt.savefig('revised eda charts/N10_top_mechanics.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Average score per top mechanic
top15_mechs = mech_sums.head(15).index.tolist()
mech_score_data = []
for col in top15_mechs:
    sub = df[df[col] == 1]
    if len(sub) > 5:
        mech_score_data.append({
            'Mechanic': col.replace('Mech_', '')[:40],
            'Count': len(sub),
            'Mean Score': sub['Average Score'].mean()
        })

mech_df = pd.DataFrame(mech_score_data).sort_values('Mean Score', ascending=False)

# CHART 11: Average score by top mechanic
fig, ax = plt.subplots(figsize=(10, 7))
colors_mech = ['#10B981' if s >= 7 else '#3B82F6' if s >= 6 else '#EF4444'
               for s in mech_df['Mean Score']]
ax.barh(range(len(mech_df)-1, -1, -1), mech_df['Mean Score'].values,
        color=colors_mech, edgecolor='white')
ax.set_yticks(range(len(mech_df)-1, -1, -1))
ax.set_yticklabels(mech_df['Mechanic'].values, fontsize=10)
ax.axvline(df['Average Score'].mean(), color='gray', linestyle='--', linewidth=1.5,
           label=f'Overall mean: {df["Average Score"].mean():.2f}')
ax.set_xlabel('Average Score', fontsize=13)
ax.set_title('Average Score by Mechanic (Top 15 Most Common)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('revised eda charts/N11_score_by_mechanic.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nAverage score by mechanic:')
print(mech_df[['Mechanic', 'Count', 'Mean Score']].to_string(index=False))

### Finding 10: Mechanics Are Strong Predictors of Quality

**Observations:**
- Mechanics like **Cooperative Game**, **Legacy Game**, **Deck Construction**, and **Variable Player Powers** are associated with higher-than-average scores — these are hallmarks of modern, well-designed hobby games
- Simple mechanics like **Roll / Spin and Move** correlate with lower scores — classic family/casual games that the BGG community rates less favorably
- The mean score variance across mechanics is notable, suggesting mechanic features could be strong predictors

**New modeling opportunity:** Including mechanic binary features alongside TF-IDF may significantly improve prediction accuracy for games with short or generic descriptions.

---
## 11. Summary: Key Findings and How They Shape Our Approach

| # | Finding | Impact on Modeling |
|---|---------|-------------------|
| 1 | ~6,000 exact duplicates → 4,883 unique records after dedup | Always deduplicate before training |
| 2 | Wider score spread than original (std=1.58 vs 0.81) | More score differentiation; regression is more powerful |
| 3 | Class imbalance reduced but still present | Ordinal regression preferred over 3-class classification |
| 4 | 10-point scale more balanced with new data | Same evaluation framework applies |
| 5 | Low-review games have noisy scores | Consider review-count weighting in training |
| 6 | Descriptions sufficient for NLP; Hit games longer | TF-IDF remains a valid feature source |
| 7 | Vocabulary is rich and thematic | Custom stopwords needed; no major change from original |
| 8 | Score extremes make sense per BGG community norms | Model is valid for hobby retail use case |
| 9 | Category features show score variation | `Cat_*` columns add structured signal missing from original |
| 10 | Mechanic features are strong quality predictors | `Mech_*` columns add the strongest new structured signal |

### New Modeling Opportunity

The original dataset only had text descriptions as features. The new encoded dataset enables:
```
Old pipeline:  TF-IDF(description) → Ridge/Lasso/kNN → RMSE
New pipeline:  TF-IDF(description) + Cat_* + Mech_* → model → RMSE
                                       ^
                              351 new binary features
```
Including structured features alongside text should improve predictive performance, especially for games with generic descriptions where mechanics/categories carry the discriminative signal.

In [ ]:
print('=' * 60)
print('QUICK REFERENCE — NEW DATASET NUMBERS')
print('=' * 60)

print(f'\nDATASET')
print(f'  Raw rows:         {len(df_raw):,}')
print(f'  After dedup:      {len(df):,}')
print(f'  Unique names:     {df["Name"].nunique():,}')
print(f'  Category columns: {len(cat_cols)}')
print(f'  Mechanic columns: {len(mech_cols)}')

print(f'\nSCORE STATS')
print(f'  Mean:   {df["Average Score"].mean():.2f}')
print(f'  Median: {df["Average Score"].median():.2f}')
print(f'  Std:    {df["Average Score"].std():.2f}')
print(f'  Min:    {df["Average Score"].min():.2f} | Max: {df["Average Score"].max():.2f}')

print(f'\nCLASS DISTRIBUTION')
for label in ['Hit (8-10)', 'Average (6-7.9)', 'Flop (<6)']:
    count = (df['label'] == label).sum()
    print(f'  {label}: {count:,} ({count/len(df)*100:.1f}%)')

print(f'\nDESCRIPTIONS')
print(f'  Mean word count:   {df["word_count"].mean():.0f} words')
print(f'  Median word count: {df["word_count"].median():.0f} words')

print(f'\nREVIEWS')
print(f'  Mean reviews:   {df["Number of Reviews"].mean():.1f}')
print(f'  Median reviews: {df["Number of Reviews"].median():.0f}')
print(f'  Max reviews:    {df["Number of Reviews"].max():,}')

print(f'\nTOP 3 HIGHEST RATED (10+ reviews)')
for _, row in df_reliable.nlargest(3, 'Average Score').iterrows():
    print(f'  {row["Name"]}: {row["Average Score"]:.2f}')

print(f'\nBOTTOM 3 LOWEST RATED (10+ reviews)')
for _, row in df_reliable.nsmallest(3, 'Average Score').iterrows():
    print(f'  {row["Name"]}: {row["Average Score"]:.2f}')

---
## 12. Chart Export Checklist

All charts saved to `revised eda charts/` at 300 DPI:

| File | Description |
|------|-------------|
| `N01_score_distribution.png` | Score histogram with mean/median lines |
| `N02_class_distribution.png` | Hit vs Average vs Flop bar chart |
| `N03_10point_distribution.png` | 10-point ordinal scale |
| `N04_reviews_analysis.png` | Review count distribution + score vs reviews |
| `N05_wordcount_analysis.png` | Word count histogram + word count by class |
| `N06_top_words.png` | Top 15 frequent words |
| `N07_top_bottom_games.png` | Highest & lowest rated (10+ reviews) |
| `N08_top_categories.png` | Top 20 most common categories |
| `N09_score_by_category.png` | Average score by top categories |
| `N10_top_mechanics.png` | Top 20 most common mechanics |
| `N11_score_by_mechanic.png` | Average score by top mechanics |